# 1. Breast segmentation with R2AttU-Net

The first stage isolates the breasts from the rest of the thermal image so that downstream models only see relevant tissue. This notebook summarises:

1. **Repeated training runs** of the tuned binary model (from `results/`, runnable).
2. **Evaluation of the best model on an external FLIR test set** of 21 images acquired separately from the training data. This part needs the private clinical images and the trained weights, so the outputs below are the ones recorded when it was run; patient images are not displayed.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS = Path("../results")
FIGURES = Path("../figures")
FIGURES.mkdir(exist_ok=True)

## Repeated runs (binary model, held-out test split)

In [2]:
runs = pd.read_csv(RESULTS / "segmentation_repeated_runs.csv")
metrics = ["dice", "iou", "precision", "recall (sensitivity)", "specificity", "auc"]
summary = runs[metrics].agg(["mean", "std"]).T.round(3)
summary

,mean,std
dice,0.783,0.060
iou,0.673,0.092
precision,0.874,0.144
recall (sensitivity),0.772,0.138
specificity,0.971,0.044
auc,0.934,0.046


## Best model on the external FLIR test set

Requires `checkpoints/best/best_model.pth` and the test set prepared by `scripts/prepare_flir_test_set.py`.

In [ ]:
import json

import torch

from thermal_bc.config import root_dir
from thermal_bc.data.thermal_dataset import ThermalDataset
from thermal_bc.models.metrics import breast_mask_from_multiclass, confusion_metrics
from thermal_bc.models.r2attunet import R2AttU_Net

config = json.loads(Path("../configs/segmentation_best.json").read_text())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = R2AttU_Net(
    img_ch=1, output_ch=3, t=config["t"], base_filters=config["base_filters"], depth=config["depth"]
)
state_dict = torch.load("../checkpoints/best/best_model.pth", map_location=device)
model.load_state_dict(state_dict)
model.to(device).eval();

In [ ]:
data_dir = root_dir() / "data" / "test_flir"
test_dataset = ThermalDataset(
    images_dir=data_dir / "images", masks_dir=data_dir / "masks", height=256, width=384
)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=config["batch_size"])
print(f"Test set size: {len(test_dataset)}")

Test set size: 21


In [ ]:
images, preds, gts = [], [], []
with torch.no_grad():
    for image, mask in test_loader:
        logits = model(image.to(device)).cpu()
        # The model predicts background / left / right breast; a pixel is breast when the
        # background channel scores below 0.5.
        preds.append(breast_mask_from_multiclass(logits))
        gts.append(mask.unsqueeze(1).float())
        images.append(image)

images, preds, gts = torch.cat(images), torch.cat(preds), torch.cat(gts)
print(confusion_metrics(preds, gts))

{'Dice': 0.8379008999322676, 'IoU': 0.7210236200023139, 'Precision': 0.8200581148487935, 'Recall': 0.856537399540282, 'F1': 0.8379008949346188, 'Specificity': 0.9730005828009404, 'AUC': 0.9147689911706304}


Pixel-wise over the whole test set the model reaches a **Dice of 0.84** (IoU 0.72, sensitivity 0.86, specificity 0.97). Per image, 14 of the 21 scans exceed a Dice of 0.8; the recorded run had three images where no breast region was predicted (Dice 0), which points to failure cases worth inspecting (e.g. atypical framing or contrast).

### Qualitative results

Input, prediction and ground truth for the first five test images (not rendered here because the images are private).

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 6))
rows = [("Input", images), ("Prediction", preds), ("Ground truth", gts)]
for row, (title, batch) in enumerate(rows):
    for col in range(5):
        axes[row, col].imshow(batch[col, 0], cmap="gray")
        axes[row, col].axis("off")
    axes[row, 0].set_title(title, loc="left")
plt.tight_layout()